<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats_v2/corrections/seance2_correction.ipynb)

# Séance 3.2 — Comparer deux groupes — hasard ou vrai écart ?

**Correction** · durée : 4h (2h de cours, 2h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer pourquoi une moyenne calculée sur un échantillon n'est jamais exacte
- construire un intervalle de confiance à 95 % par rééchantillonnage
- comparer deux groupes avec un test t et lire sa p-value
- distinguer « pas de différence » de « pas de différence détectable »
- repérer les deux pièges qui rendent un test faux : dépendance des observations et tests répétés

## Exercice — Les prix baissent-ils vraiment ?

## La question

> *« Immobilier : les prix parisiens ont reculé de 5 % en 2024. »*

Le titre est dans tous les journaux. Il est exact, et vous allez le vérifier
vous-même en deux lignes.

Puis votre client du 6e appelle. Il lit la presse, il possède un appartement
rue de Vaugirard, et il pose la seule question qui l'intéresse :

> *« Et chez moi ? »*

Elle est plus difficile qu'elle n'en a l'air. Paris entier, c'est 52 941
ventes sur deux ans : à cette échelle, un écart de 5 % ne doit rien au hasard.
Le 6e, c'est 699 ventes en 2023 et 709 en 2024. Sur si peu, une moyenne
bouge toute seule d'une année sur l'autre, même si rien n'a changé.

Tout l'exercice consiste à savoir **quand un écart mesuré mérite qu'on y
croie**. À la fin, vous aurez :

- l'écart réel sur Paris, et la fourchette dans laquelle il se situe ;
- le verdict des vingt arrondissements, et la liste de ceux où l'on ne peut rien dire ;
- et la réponse au client du 6e, qui n'est ni oui ni non.

## Comment ça marche

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* nomme les outils dont vous avez besoin. Vous n'avez
rien à deviner : vous avez à appliquer le cours à ce fichier.

Vous rencontrerez aussi trois autres sortes de cellules :

- des **cellules à exécuter telles quelles** : le code est déjà écrit, il vous montre quelque chose ;
- des **cellules de vérification**, aux moments où une erreur fausserait la suite. Elles affichent `OK` ou `A REVOIR` avec un indice ;
- des **cellules de prédiction** : on vous demande d'écrire ce que vous attendez, en commentaire, *avant* d'exécuter la cellule suivante.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

## Partie 0 — Mise en route

Une ligne de plus que d'habitude dans la cellule de setup : `from scipy import
stats`, vu au cours 3.2. C'est le module qui contient les tests.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "immo_paris_2023_2024.csv")

print(ventes.shape)
ventes.head(3)

Le même fichier que la semaine dernière, avec une colonne de plus, `annee`, et
les ventes de **2023** ajoutées à celles de 2024. Le nettoyage est identique
d'une année sur l'autre : c'est le pipeline que vous aviez écrit, rejoué sur
le millésime précédent.

52 941 ventes en tout. Une ligne, un appartement, une vente.

---

## Partie 1 — La baisse annoncée

### Exercice 1 — Vérifier le titre du journal

Construisez `resume` : pour chaque année, l'effectif, la moyenne et la médiane
de `prix_m2`. Puis mettez dans `ecart_reel` l'écart des **moyennes**
(2024 moins 2023), et dans `ecart_pct` le même écart en pourcentage de 2023.

> **Rappel.** `groupby("annee")` puis `agg` avec la liste des indicateurs.
> `resume.loc[2024, "mean"]` va chercher une case par son étiquette de ligne
> puis son nom de colonne.

In [ ]:
resume = ventes.groupby("annee")["prix_m2"].agg(["count", "mean", "median"])
print(resume.round(1))

ecart_reel = resume.loc[2024, "mean"] - resume.loc[2023, "mean"]
ecart_pct = 100 * ecart_reel / resume.loc[2023, "mean"]

print("ecart :", round(ecart_reel, 1), "euros par m2, soit", round(ecart_pct, 1), "%")

In [ ]:
verifier("l'ecart en euros", round(ecart_reel, 1) == -548.7, "moyenne 2024 moins moyenne 2023, dans cet ordre : le resultat est negatif")
verifier("l'ecart en pourcentage", round(ecart_pct, 1) == -5.1, "l'ecart rapporte a la moyenne de 2023, pas a celle de 2024")

**-5,1 %.** Le journal a raison : le prix moyen du mètre carré parisien est
passé de 10 750 à 10 201 €, soit **549 € de moins**. La médiane
confirme, avec un recul du même ordre.

La cellule suivante isole les deux colonnes qu'on va comparer pendant tout
l'exercice. Exécutez-la.

In [ ]:
paris23 = ventes.query("annee == 2023")["prix_m2"]
paris24 = ventes.query("annee == 2024")["prix_m2"]

print(len(paris23), "ventes en 2023 |", len(paris24), "en 2024")

---

## Partie 2 — Ressentir : les agences de quartier

Une agence de quartier n'a pas 52 941 ventes sous les yeux. Elle en a vu
cinquante l'an dernier, cinquante cette année, celles qu'elle a faites
elle-même.

**Question, avant tout calcul : cette agence-là aurait-elle vu la baisse ?**

### Tirer un échantillon

`sample(50, random_state=0)` prend cinquante lignes au hasard. Au cours 3.2,
vous écriviez `sample(len(fr), replace=True)` : la remise était nécessaire
parce qu'on tirait autant de valeurs qu'il y en avait, pour rejouer un
échantillon à partir de lui-même.

Ici, c'est autre chose : on tire cinquante ventes parmi les 25 209 de 2024
pour simuler une agence qui n'en a vu que cinquante. Il y a de quoi faire, la remise ne
sert à rien. `random_state` garde son rôle habituel : il fixe le tirage pour
que tout le monde obtienne les mêmes cinquante ventes.

### Exercice 2 — Une agence

Tirez cinquante ventes de chaque année avec `random_state=0`, rangez-les dans
`agence23` et `agence24`, et calculez l'écart de leurs moyennes dans
`ecart_agence`.

> **Rappel.** `serie.sample(50, random_state=0)`, puis `mean()` sur chacune.
> L'écart se calcule dans le même sens qu'à l'exercice 1 : 2024 moins 2023.

In [ ]:
agence23 = paris23.sample(50, random_state=0)
agence24 = paris24.sample(50, random_state=0)

ecart_agence = agence24.mean() - agence23.mean()

print("cette agence a mesure", round(ecart_agence), "euros par m2")

In [ ]:
verifier("l'ecart de l'agence", round(ecart_agence) == -259, "sample(50, random_state=0) sur chaque annee, puis la difference des moyennes")

Cette agence-là a mesuré **-259 €** là où Paris en perdait 549.
Changeons d'agence : la cellule suivante rejoue le tirage avec trois autres
numéros.

In [ ]:
for graine in [1, 2, 3]:
    e = (paris24.sample(50, random_state=graine).mean()
         - paris23.sample(50, random_state=graine).mean())
    print(f"agence numero {graine} : {e:>8.0f} euros par m2")

Trois agences, trois réponses, et elles ne sont même pas toutes du même signe.
Aucune n'a tort : chacune a correctement calculé la moyenne de ce qu'elle a
vu. C'est le tirage qui change, pas le calcul.

### Exercice 3 — Trois cents agences

Mettez ce tirage dans une boucle de 300 tours, en faisant varier
`random_state` de 0 à 299. Rangez les écarts dans une liste, transformez-la en
Series appelée `agences`, puis calculez dans `part_hausses` le pourcentage
d'agences qui mesurent une **hausse**.

Tracez enfin l'histogramme, avec un trait rouge sur l'écart réel.

> **Rappel.** Une liste vide avant la boucle, `append` dedans,
> `pd.Series(ma_liste)` après : c'est le réflexe du cours 3.2. Une comparaison
> comme `agences > 0` rend des Vrai/Faux, et leur `mean()` est une proportion.
>
> Pour le trait : `plt.axvline(valeur, color="#D64541", linewidth=2.5)`.

In [ ]:
ecarts = []
for i in range(300):
    ecarts.append(paris24.sample(50, random_state=i).mean()
                  - paris23.sample(50, random_state=i).mean())

agences = pd.Series(ecarts)
part_hausses = round(100 * (agences > 0).mean(), 1)

print("agences qui mesurent une hausse :", part_hausses, "%")

agences.plot(kind="hist", bins=40, figsize=(7, 4), color="grey")
plt.axvline(ecart_reel, color="#D64541", linewidth=2.5, label="la vraie baisse")
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("ecart mesure par l'agence (euros par m2)")
plt.ylabel("nombre d'agences")
plt.title("300 agences de quartier, 50 ventes par an chacune")
plt.legend()
plt.show()

In [ ]:
verifier("trois cents agences", len(agences) == 300, "range(300), un append par tour")
verifier("la part de hausses", part_hausses == 23.7, "(agences > 0).mean() rend une proportion : x 100 pour des %")

**23,7 % des agences mesurent une hausse** alors que Paris baisse de 5 %.
Le trait rouge est la vérité ; le tas s'étale de part et d'autre, et il déborde
largement du côté positif. Une agence sur quatre aurait dit à ses clients que
les prix montaient, en toute bonne foi, chiffres à l'appui.

Vous connaissez la vérité parce que vous avez les 52 941 ventes. **Une
agence qui n'a que son échantillon ne sait pas où elle tombe dans cet
histogramme.** C'est exactement le problème que le reste de l'exercice
résout : quand on n'a qu'un échantillon, comment savoir ce que vaut le chiffre
qu'on en tire ?

---

## Partie 3 — Mesurer : la fourchette, puis le test

Au cours 3.2, vous avez encadré **une** moyenne en la rejouant mille fois.
Ici on veut encadrer **un écart** entre deux moyennes. Le principe ne change
pas : à chaque tour, on rejoue les deux années, et on note l'écart obtenu.

### Exercice 4 — La fourchette de l'écart parisien

Faites mille tours. À chaque tour, tirez avec remise autant de ventes qu'il y
en a dans chaque année, et rangez l'écart des moyennes dans une liste.
Transformez la liste en Series `boot`, puis lisez l'intervalle de confiance à
95 % dans `ic_bas` et `ic_haut`.

Comptez une dizaine de secondes d'exécution : mille tirages sur 52 941
lignes, cela fait du travail.

> **Rappel.** Le rééchantillonnage du cours : `serie.sample(len(serie),
> replace=True, random_state=i)`. L'intervalle à 95 %, ce sont les quantiles
> 0.025 et 0.975 de la Series des mille écarts.

In [ ]:
ecarts_boot = []
for i in range(1000):
    b23 = paris23.sample(len(paris23), replace=True, random_state=i)
    b24 = paris24.sample(len(paris24), replace=True, random_state=i)
    ecarts_boot.append(b24.mean() - b23.mean())

boot = pd.Series(ecarts_boot)
ic_bas = boot.quantile(0.025)
ic_haut = boot.quantile(0.975)

print("ecart plausible : de", round(ic_bas), "a", round(ic_haut), "euros par m2")

In [ ]:
verifier("mille tirages", len(boot) == 1000, "range(1000), un append par tour")
verifier("la borne basse", abs(ic_bas - (-607)) < 25, "quantile(0.025) sur la Series des ecarts")
verifier("la borne haute", abs(ic_haut - (-490)) < 25, "quantile(0.975) sur la Series des ecarts")
verifier("la fourchette ne contient pas zero", ic_haut < 0, "si votre borne haute est positive, verifiez le sens de la soustraction")

**De -607 à -490 € par m².** Les mille rejeux donnent tous une baisse,
et la fourchette est étroite : 117 € de large, pour un écart de 549.
**Zéro n'est nulle part dedans.** Sur Paris entier, la baisse n'est pas une
question d'échantillon.

### Exercice 5 — La même fourchette, pour l'agence

Refaites exactement le même travail sur `agence23` et `agence24`, les
cinquante ventes de l'exercice 2. Rangez les bornes dans `ic_agence_bas` et
`ic_agence_haut`.

Ici, la remise est indispensable : on rejoue un échantillon de cinquante à
partir de lui-même, c'est le cas du cours.

> **Rappel.** Le même bloc que ci-dessus, avec `agence23` et `agence24` à la
> place de `paris23` et `paris24`.

In [ ]:
petits = []
for i in range(1000):
    petits.append(agence24.sample(len(agence24), replace=True, random_state=i).mean()
                  - agence23.sample(len(agence23), replace=True, random_state=i).mean())

boot_agence = pd.Series(petits)
ic_agence_bas = boot_agence.quantile(0.025)
ic_agence_haut = boot_agence.quantile(0.975)

print("l'agence peut seulement dire : entre", round(ic_agence_bas),
      "et", round(ic_agence_haut), "euros par m2")

In [ ]:
verifier("la borne basse de l'agence", abs(ic_agence_bas - (-1907)) < 120, "le meme bootstrap, sur agence23 et agence24")
verifier("la fourchette de l'agence contient zero", ic_agence_bas < 0 < ic_agence_haut,
         "avec 50 ventes par an, la hausse reste plausible : les deux bornes doivent encadrer zero")

**De -1 907 à 1 414 €.** La fourchette de l'agence est
**28 fois plus large** que celle de Paris, et surtout **elle contient
zéro** : à partir de ses cinquante ventes, une hausse reste parfaitement
plausible. L'agence n'a pas mesuré une baisse. Elle a mesuré quelque chose
qui pourrait être une baisse, une stagnation, ou une hausse.

C'est la réponse à la question de la partie 2. Une agence ne sait pas où elle
tombe dans l'histogramme, mais elle peut calculer **la largeur** de son
incertitude, et cette largeur lui dit de se taire.

### Exercice 6 — Le test, la même question en un nombre

Le test t répond à : *si rien n'avait bougé entre 2023 et 2024, à quelle
fréquence observerait-on un écart au moins aussi grand que celui-ci ?*

Calculez la p-value sur Paris entier dans `p_paris`, puis sur l'agence dans
`p_agence`. Affichez les deux.

> **Rappel.** `stats.ttest_ind(a, b, equal_var=False).pvalue`. L'argument
> `equal_var=False` n'est pas le comportement par défaut : il se remet à
> chaque appel.

In [ ]:
p_paris = stats.ttest_ind(paris24, paris23, equal_var=False).pvalue
p_agence = stats.ttest_ind(agence24, agence23, equal_var=False).pvalue

print("Paris entier :", p_paris)
print("l'agence     :", round(p_agence, 3))

In [ ]:
verifier("la p-value de Paris", p_paris < 0.001, "ttest_ind sur paris24 et paris23, avec equal_var=False")
verifier("la p-value de l'agence", round(p_agence, 2) == 0.77, "ttest_ind sur agence24 et agence23")

**2 × 10⁻⁷⁴ contre 0,77.** Deux nombres, deux mondes.

Sur Paris, la p-value s'écrit avec soixante-treize zéros après la virgule : si
les prix n'avaient pas bougé, un écart de cette taille ne s'observerait
jamais. Sur l'agence, 0,77 veut dire que l'écart qu'elle a mesuré
s'observerait dans 77 % des cas alors même que rien n'aurait bougé.
On ne conclut rien.

> ⚠️ Et « on ne conclut rien » n'est pas « les prix n'ont pas bougé ». Les prix
> ont bougé : vous le savez, vous avez le fichier entier. L'agence, elle, ne
> peut pas le savoir avec cinquante ventes.

---

## Partie 4 — Vingt arrondissements, vingt verdicts

Reste la question du client. On va la poser vingt fois.

### Un tableau construit à partir de listes

Au bloc 2, une boucle remplissait deux listes, et `pd.Series(valeurs,
index=etiquettes)` les réunissait. Quand la boucle produit **plusieurs**
grandeurs par tour, l'objet qui convient est un DataFrame :

```python
tableau = pd.DataFrame({"colonne_a": liste_a, "colonne_b": liste_b}, index=liste_etiquettes)
```

Un dictionnaire : à gauche le nom de chaque colonne, à droite la liste qui la
remplit. Les listes doivent avoir la même longueur, et `index` donne les
étiquettes de ligne.

### Exercice 7 — Le tableau des vingt

Écrivez une boucle sur `range(1, 21)`. À chaque tour, isolez les ventes de
l'arrondissement pour chaque année, puis rangez dans trois listes : le numéro,
l'écart des moyennes **en pourcentage de 2023**, et la p-value du test.

Construisez ensuite `verdict`, avec les colonnes `ecart_pct` et `p`, indexé
par le numéro d'arrondissement.

> **Rappel.** Dans une chaîne `query`, `@arr` désigne la variable `arr` de la
> boucle — c'est le `@` du bloc 2. Deux conditions se combinent avec `and`
> dans la même chaîne.

In [ ]:
numeros = []
pourcentages = []
valeurs_p = []

for arr in range(1, 21):
    x23 = ventes.query("annee == 2023 and arrondissement == @arr")["prix_m2"]
    x24 = ventes.query("annee == 2024 and arrondissement == @arr")["prix_m2"]
    numeros.append(arr)
    pourcentages.append(100 * (x24.mean() - x23.mean()) / x23.mean())
    valeurs_p.append(stats.ttest_ind(x24, x23, equal_var=False).pvalue)

verdict = pd.DataFrame({"ecart_pct": pourcentages, "p": valeurs_p}, index=numeros)

print(verdict.round(3))

In [ ]:
verifier("vingt lignes", len(verdict) == 20, "la boucle va de 1 a 20 : range(1, 21)")
verifier("l'ecart du 8e", round(verdict.loc[8, "ecart_pct"], 1) == -0.3, "l'ecart rapporte a la moyenne 2023 de cet arrondissement")
verifier("l'ecart du 19e", round(verdict.loc[19, "ecart_pct"], 1) == -9.0, "le 19e est celui qui recule le plus")

**Les vingt arrondissements affichent une baisse.** Pas dix-sept, pas
dix-huit : les vingt. Reste à savoir lesquelles tiennent.

### Exercice 8 — Lesquelles tiennent ?

Ajoutez à `verdict` une colonne `significatif`, vraie quand la p-value est
sous 0,05. Comptez-les dans `nb_signif`, et mettez dans `non_concluants` la
liste des arrondissements où l'on ne peut pas conclure.

> **Rappel.** Une comparaison sur une colonne rend une colonne de Vrai/Faux, et
> `sum()` compte les Vrai. `query("significatif == False")` garde les lignes
> voulues, `list(...index)` en extrait les étiquettes.

In [ ]:
verdict["significatif"] = verdict["p"] < 0.05

nb_signif = verdict["significatif"].sum()
non_concluants = list(verdict.query("significatif == False").index)

print(nb_signif, "arrondissements sur 20")
print("on ne peut pas conclure dans le", non_concluants)

In [ ]:
verifier("le nombre de baisses retenues", nb_signif == 16, "la colonne p comparee a 0.05, puis sum()")
verifier("les quatre autres", non_concluants == [1, 2, 6, 8], "les lignes ou significatif vaut False")

**16 sur 20.** Les quatre exceptions sont le 1er, le 2e, le 6e et le 8e — et elles ne
racontent pas la même histoire. Comparez ces deux lignes :

| | ventes 2023 | ventes 2024 | écart | p |
|---|---|---|---|---|
| **8e** | 527 | 599 | -0,3 % | 0,87 |
| **6e** | 699 | 709 | -1,6 % | 0,32 |

Dans le **8e**, l'écart mesuré est de -0,3 %, c'est-à-dire rien. Les prix n'y
ont vraisemblablement pas bougé, et le test le dit sans ambiguïté.

Dans le **6e**, l'écart mesuré est de -1,6 %, du même ordre que dans des
arrondissements où la baisse est retenue. Ce qui manque n'est pas l'écart,
c'est le nombre de ventes : 699 et 709, contre plus de deux mille dans
le 15e. La fourchette est trop large pour exclure zéro.

**Deux situations, une seule phrase dans le tableau.** C'est pourquoi « non
significatif » ne se traduit jamais par « pas de baisse » sans regarder
l'effectif et la taille de l'écart.

### Exercice 9 — Vingt tests, et le piège du cours

Le cours 3.2 finit sur une mise en garde : lancer vingt tests sur des groupes
où il n'y a rien à trouver produit en moyenne un « significatif » quand même,
et une chance sur deux d'en produire au moins un. Vous venez de lancer
**exactement vingt tests**.

La correction de Bonferroni divise le seuil par le nombre de tests. Calculez
`seuil` puis `nb_bonf`, le nombre d'arrondissements qui passent ce seuil-là,
et `perdus`, la liste de ceux qui étaient retenus à 0,05 et ne le sont plus.

> **Rappel.** Le seuil corrigé, c'est 0,05 divisé par le nombre de tests.
> Pour `perdus` : les lignes dont la p-value dépasse le nouveau seuil parmi
> celles qui étaient significatives.

In [ ]:
seuil = 0.05 / 20

nb_bonf = (verdict["p"] < seuil).sum()
perdus = list(verdict.query("p >= @seuil and significatif == True").index)

print("seuil corrige :", seuil)
print(nb_bonf, "arrondissements tiennent | perdus :", perdus)

In [ ]:
verifier("le seuil corrige", round(seuil, 4) == 0.0025, "0.05 divise par le nombre de tests, c'est-a-dire 20")
verifier("les baisses qui tiennent", nb_bonf == 14, "la colonne p comparee au nouveau seuil")
verifier("ceux qui tombent", perdus == [3, 7], "significatifs a 0,05 mais pas au seuil corrige")

**14 au lieu de 16.** Le 3e et le 7e passaient de justesse, ils
ne passent plus. Leur baisse n'a pas disparu pour autant : ce qui a changé,
c'est l'exigence, parce qu'on a posé vingt questions au lieu d'une.

Faut-il corriger ici ? Le cours donne la règle : **la question d'abord, le test
ensuite.** Si vous aviez décidé à l'avance d'examiner le 6e, parce qu'un client
vous l'a demandé, un seul test a été posé et le seuil de 0,05 est le bon. Si
vous parcourez les vingt pour voir « lesquels ressortent », alors vous faites
une campagne, et le seuil corrigé s'impose.

La cellule suivante met les vingt verdicts en image. Le petit `if` dans la
liste des couleurs se lit à voix haute : rouge si significatif, gris sinon.

In [ ]:
v = verdict.sort_values("ecart_pct")
couleurs = ["#D64541" if oui else "grey" for oui in v["significatif"]]

plt.figure(figsize=(7, 6))
plt.barh([f"{a}e" for a in v.index], v["ecart_pct"], color=couleurs)
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("evolution du prix moyen au m2, 2023 vers 2024 (%)")
plt.title("Rouge : baisse retenue. Gris : on ne peut pas conclure.")
plt.tight_layout()
plt.show()

---

## Partie 5 — Combien de ventes aurait-il fallu ?

L'agence de la partie 2 n'a pas vu la baisse. Était-elle mal outillée, ou
est-ce qu'aucune agence de cette taille n'y serait arrivée ?

On peut répondre exactement, puisqu'on connaît la vérité : on rejoue trois
cents agences, on leur fait passer le test, et on compte combien concluent.

### Exercice 10 — Le taux de réussite d'une agence de 200 ventes

Reprenez la boucle de l'exercice 3, avec `n = 200` ventes par année cette
fois, et ajoutez le test à chaque tour. Comptez dans `part_200` le pourcentage
d'agences qui **concluent à une baisse** : p-value sous 0,05 **et** moyenne
2024 inférieure à celle de 2023.

> **Rappel.** Un compteur à zéro avant la boucle, `+= 1` quand les deux
> conditions sont remplies. Deux conditions se combinent avec `and`.

In [ ]:
n = 200
conclusions = 0

for i in range(300):
    x23 = paris23.sample(n, random_state=i)
    x24 = paris24.sample(n, random_state=i)
    p = stats.ttest_ind(x24, x23, equal_var=False).pvalue
    if p < 0.05 and x24.mean() < x23.mean():
        conclusions += 1

part_200 = round(100 * conclusions / 300, 1)
print(part_200, "% des agences de 200 ventes par an concluent a une baisse")

In [ ]:
verifier("le taux de reussite a 200 ventes", part_200 == 29.7,
         "compter les tours ou p < 0.05 ET la moyenne 2024 est la plus basse")

Et pour les autres tailles ? La cellule suivante reprend votre boucle et la
fait tourner pour quatre tailles d'agence. Comptez une dizaine de secondes.

In [ ]:
for n in [20, 50, 200, 1000]:
    conclusions = 0
    for i in range(300):
        x23 = paris23.sample(n, random_state=i)
        x24 = paris24.sample(n, random_state=i)
        p = stats.ttest_ind(x24, x23, equal_var=False).pvalue
        if p < 0.05 and x24.mean() < x23.mean():
            conclusions += 1
    print(f"{n:>5} ventes par an : {100 * conclusions / 300:>5.1f} % concluent a une baisse")

| Ventes par an | Agences qui concluent à une baisse |
|---|---|
| 20 | 5,0 % |
| 50 | 10,7 % |
| 200 | 29,7 % |
| 1 000 | 94,3 % |

**Avec vingt ventes par an, 5,0 %.** Dix-neuf agences sur vingt ne voient
rien, alors que la baisse est bien là et qu'elle vaut 5 %. Il faut monter à
mille ventes par année, soit le volume annuel d'un arrondissement entier, pour
que la baisse ressorte neuf fois sur dix.

Ce taux de réussite porte un nom : c'est la **puissance** du test. Elle dépend
de trois choses — la taille de l'écart qu'on cherche, la dispersion des
données, et le nombre d'observations. Les deux premières sont données ; la
troisième est la seule sur laquelle on peut agir.

Et elle se calcule **avant** de lancer l'étude. C'est ce qui permet de dire à
un client « avec vos cinquante ventes, je ne pourrai pas répondre » plutôt que
de lui rendre un résultat qui ne veut rien dire.

L'agence de la partie 2 n'était donc pas mal outillée. Elle n'avait simplement
pas assez de ventes, et aucune méthode n'aurait rattrapé ça.

---

## Pour conclure

### La réponse au client du 6e

Complétez cette cellule de texte (double-clic pour l'éditer) en trois phrases :

- Ce qu'on sait de Paris : …
- Ce qu'on ne sait pas du 6e, et pourquoi : …
- Ce qu'il faudrait pour le savoir : …

### Une réserve, à dire aussi

Tout l'exercice compare **les ventes de 2023 aux ventes de 2024**, et non les
mêmes appartements vendus deux fois. Ce ne sont pas les mêmes biens.

Souvenez-vous de ce que vous avez trouvé à l'exercice précédent : les grands
appartements sont concentrés dans les arrondissements chers, et ils se vendent
plus cher au mètre carré. Si, en 2024, il s'est vendu un peu moins de grands
appartements, le prix moyen au m² baisse **sans qu'aucun prix n'ait bougé**.

On ne traitera pas cette question ici : elle demande de comparer des biens
comparables, ce qui est l'objet de la suite du bloc. Mais un analyste la
mentionne, parce que quelqu'un la posera.

## Ce que vous avez fait

- vous avez vérifié le titre du journal, et il était exact ;
- vous avez montré, en trois cents tirages, qu'une agence de quartier pouvait mesurer l'inverse de la réalité ;
- vous avez encadré l'écart parisien, et celui de l'agence, par la même méthode ;
- vous avez rendu vingt verdicts, et vous savez lesquels ne sont pas des verdicts ;
- vous avez chiffré ce qu'il faut de ventes pour pouvoir conclure.

| Vous avez utilisé | Pour |
|---|---|
| `groupby("annee").agg([...])` | les deux millésimes côte à côte |
| `sample(n, random_state=i)` | simuler une agence qui n'a vu que n ventes |
| une boucle, une liste, `pd.Series(...)` | trois cents agences, et la forme du doute |
| `sample(len(s), replace=True)` | le rééchantillonnage du cours, sur un écart cette fois |
| `quantile(0.025)`, `quantile(0.975)` | l'intervalle de confiance à 95 % |
| `stats.ttest_ind(a, b, equal_var=False)` | la même question en un nombre |
| `pd.DataFrame({...}, index=...)` | un tableau construit à partir de listes |
| `query("... == @variable")` | les vingt sous-tableaux de la boucle |
| `0.05 / 20` | le seuil corrigé, parce qu'on a posé vingt questions |
| un compteur dans une boucle | la puissance, mesurée plutôt que supposée |

### Les trois phrases à retenir

1. **Un écart mesuré n'est pas un écart réel.** Entre les deux, il y a la
   taille de l'échantillon, et elle se calcule.
2. **« On ne peut pas conclure » n'est pas « il n'y a pas d'écart ».** Le 6e et
   le 8e sont tous les deux non significatifs, pour deux raisons opposées.
3. **Vingt tests ne s'interprètent pas comme un seul.** La question se décide
   avant de regarder les données.

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.